In [ ]:
!pip install -q git+https://github.com/annyms7519/gwoet.git

In [ ]:
import gwoet
print(gwoet.__version__)

In [ ]:
from sklearn.datasets import fetch_openml

X, y = fetch_openml(
    data_id=31,  # credit-g
    as_frame=True,
    return_X_y=True,
    n_retries=10,
    delay=10,
)

display(X.head())
print(y)

# Discrete Naive Bayes

In [ ]:
import matplotlib.pyplot as plt
import pprint
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = gwoet.DiscreteNaiveBayes()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"y_true = {y_test.to_numpy()}")
print(f"y_pred = {y_pred}")

acc_train = model.score(X_train, y_train)
acc_test = model.score(X_test, y_test)

print(f"accuracy (train): {acc_train:.3f}")
print(f"accuracy (test):  {acc_test:.3f}")

p = model.predict_proba(X_test)
auc_score = roc_auc_score(y_test.to_numpy(), p[:,1])
print(f"roc_auc_score     {auc_score:.3f}")

RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.plot([0, 1], [0, 1], color="k", linestyle="--")
plt.grid()
plt.show()

iv = model.get_iv()
print("Inforvation Value (IV) =")
pprint.pp(iv)

model.plot_woe()
None

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = gwoet.DiscreteNaiveBayes()

scores = cross_val_score(model, X, y, cv=cv)
print(f"accuracy (5-fold cv): {np.round(scores, decimals = 3)}", end = "")
print(f", mean = {scores.mean():.3f}, stdev = {scores.std():.3f}")

cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"auc      (5-fold cv): {np.round(cv_auc, decimals = 3)}", end = "")
print(f", mean = {cv_auc.mean():.3f}, stdev = {cv_auc.std():.3f}")


# Graphical WoE Transformer

## GWoET+DNB

In [ ]:
import matplotlib.pyplot as plt
import pprint
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

tfm = gwoet.GraphicalWoETransformer(
    l1_selection=False,
    target_degree='adaptive',
    random_state=0,
    verbose=True,
)
tfm.fit(X_train, y_train)

plt.figure(figsize=(12, 8))
tfm.draw(node_size=100, font_size=10)

X_train_new = tfm.transform(X_train)
X_test_new = tfm.transform(X_test)
print(f"num of variables: {len(X_train_new.columns)}")
display(X_train_new.head())

model = gwoet.DiscreteNaiveBayes()
model.fit(X_train_new, y_train)

y_pred = model.predict(X_test_new)
print(f"y_true = {y_test.to_numpy()}")
print(f"y_pred = {y_pred}")

acc_train = model.score(X_train_new, y_train)
acc_test = model.score(X_test_new, y_test)

print(f"accuracy (train): {acc_train:.3f}")
print(f"accuracy (test):  {acc_test:.3f}")

p = model.predict_proba(X_test_new)
auc_score = roc_auc_score(y_test.to_numpy(), p[:,1])
print(f"roc_auc_score     {auc_score:.3f}")

RocCurveDisplay.from_estimator(model, X_test_new, y_test)
plt.plot([0, 1], [0, 1], color="k", linestyle="--")
plt.grid()
plt.show()

iv = model.get_iv()
print("Inforvation Value (IV) =")
pprint.pp(iv)

model.plot_iv()
model.plot_woe()
None

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = Pipeline([
    (
        "gwoet",
        gwoet.GraphicalWoETransformer(
            l1_selection=False,
            target_degree="adaptive",
            random_state=0,
        ),
    ),
    ("classifier", gwoet.DiscreteNaiveBayes()),
])

scores = cross_val_score(model, X, y, cv=cv)
print(f"accuracy (5-fold cv): {np.round(scores, decimals = 3)}", end = "")
print(f", mean = {scores.mean():.3f}, stdev = {scores.std():.3f}")

cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"auc      (5-fold cv): {np.round(cv_auc, decimals = 3)}", end = "")
print(f", mean = {cv_auc.mean():.3f}, stdev = {cv_auc.std():.3f}")

## GWoET+L1

In [ ]:
import matplotlib.pyplot as plt
import pprint
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector as selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

tfm = gwoet.GraphicalWoETransformer(
    l1_selection=False,
    target_degree='adaptive',
    random_state=0,
    verbose=True,
)
tfm.fit(X_train, y_train)

plt.figure(figsize=(12, 8))
tfm.draw(node_size=100, font_size=10)

X_train_new = tfm.transform(X_train)
X_test_new = tfm.transform(X_test)
print(f"num of variables: {len(X_train_new.columns)}")
display(X_train_new.head())

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)
column_transformer = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            selector(dtype_exclude=["object", "category"]),
        ),
        (
            "cat",
            categorical_transformer,
            selector(dtype_include=["object", "category"]),
        ),
    ]
)

X_train_new2 = column_transformer.fit_transform(X_train_new)
X_test_new2 = column_transformer.transform(X_test_new)

model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=1.0,
    random_state=0,
)

model.fit(X_train_new2, y_train)

y_pred = model.predict(X_test_new2)
print(f"y_true = {y_test.to_numpy()}")
print(f"y_pred = {y_pred}")

acc_train = model.score(X_train_new2, y_train)
acc_test = model.score(X_test_new2, y_test)

print(f"accuracy (train): {acc_train:.3f}")
print(f"accuracy (test):  {acc_test:.3f}")

p = model.predict_proba(X_test_new2)
auc_score = roc_auc_score(y_test.to_numpy(), p[:,1])
print(f"roc_auc_score     {auc_score:.3f}")

RocCurveDisplay.from_estimator(model, X_test_new2, y_test)
plt.plot([0, 1], [0, 1], color="k", linestyle="--")
plt.grid()
plt.show()

coef = gwoet.make_l1_coefficient_dict(
    model,
    column_transformer,
    X_train_new,
)

gwoet.plot_l1_coefficients(
    coef,
    exclude_zero=False,
)

feature_names = (
    column_transformer
    .get_feature_names_out()
)
# Remove prefixes such as "num__" and "cat__"
feature_names = [
    name.split("__", 1)[-1]
    for name in feature_names
]

coef_table = gwoet.plot_l1_feature_coefficients(
    model.coef_[0],
    feature_names,
    top_n=20,
    exclude_zero=True,
)

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector as selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)
column_transformer = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            selector(dtype_exclude=["object", "category"]),
        ),
        (
            "cat",
            categorical_transformer,
            selector(dtype_include=["object", "category"]),
        ),
    ]
)

model = Pipeline([
    (
        "gwoet",
        gwoet.GraphicalWoETransformer(
            l1_selection=False,
            target_degree='adaptive',
            random_state=0,
        ),
    ),
    ("preprocessor", column_transformer),
    (
        "classifier",
        LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=1.0,
            random_state=0,
        ),
    ),
])

scores = cross_val_score(model, X, y, cv=cv)
print(f"accuracy (5-fold cv): {np.round(scores, decimals = 3)}", end = "")
print(f", mean = {scores.mean():.3f}, stdev = {scores.std():.3f}")

cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"auc      (5-fold cv): {np.round(cv_auc, decimals = 3)}", end = "")
print(f", mean = {cv_auc.mean():.3f}, stdev = {cv_auc.std():.3f}")